# Final Dataset Construction

In [ ]:
import pandas as pd
import ast
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
def safe_literal_eval(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return []

def select_CA(ca_positions, n_authors):
    if not ca_positions:  # no CA, just in case
        return None, None
    if 0 in ca_positions:
        return 0, 'First'
    elif (n_authors - 1) in ca_positions:
        return n_authors - 1, 'Last'
    else:
        return ca_positions[0], 'Middle'
    
def priority(row):
    if pd.notna(row['APC_Accounting_Value']):
        return row['APC_Accounting_Value'], 'APC_Accounting_Value'
    if pd.notna(row['euro']):
        return row['euro'], 'Open APC'
    if pd.notna(row['median']):
        return row['median'], 'Open APC Median'
    else:
        return row['value_usd_to_eur'], 'APC_paid'
        
def price(row):
    if 0 in row['french_position']:
        return row['First_pct'] * row['price_var']
    if row['len'] - 1 in row['french_position']:
        return row['Last_pct'] * row['price_var']
    else:
        return len(row['french_position']) * row['price_var'] * row['len'] / (row['len'] - 2)

## CA Identification

### Identified CA for BSO and Couperin as a base

In [3]:
df_initial = pd.read_csv('../data/processed/initial_dataset.csv')
df_bso = df_initial[(df_initial.BSO == True) & (df_initial.has_fr_corresponding == True)]
df_couperin = df_initial[df_initial.Is_Couperin == True]
df_corr_fr = pd.concat([df_bso, df_couperin]).drop_duplicates(subset = 'doi').reset_index(drop = True)
df_corr_fr['corresponding'] = df_corr_fr['corresponding'].apply(safe_literal_eval)
df_corr_fr['len'] = df_corr_fr['corresponding'].apply(lambda x: len(x))
df_corr_fr['CA_positions'] = df_corr_fr['corresponding'].apply(lambda lst: [i for i, v in enumerate(lst) if v] if any(lst) else np.nan)
print('Initial number of records to infer:', df_corr_fr.doi.nunique())
df_corr_fr = df_corr_fr.dropna(subset = ['CA_positions']).reset_index(drop = True)
print('Initial number of records with CA:', df_corr_fr.doi.nunique())
print('Multiple corresponding records:', len(df_corr_fr[df_corr_fr.CA_positions.apply(lambda x: len(x)) > 1]))
df_corr_fr[['keep_CA_position', 'position']] = df_corr_fr.apply(lambda x: pd.Series(select_CA(x['CA_positions'], x['len'])), axis = 1)
priority_map = {'First': 3, 'Last': 2, 'Middle': 1}
df_corr_fr['priority'] = df_corr_fr['position'].map(priority_map)
df_corr_fr

Initial number of records to infer: 295391
Initial number of records with CA: 295311
Multiple corresponding records: 11920


,doi,publication_year,language,issn_l,oa_status,apc_paid,corresponding,countries,has_fr_corresponding,BSO,...,Deal_Name,Deal_Type,Eligible_Deal,APC_Accounting_Value,Is_Couperin,len,CA_positions,keep_CA_position,position,priority
0,10.3389/fnins.2013.00267,2013,en,1662-453X,gold,"{'value': 2950, 'currency': 'USD', 'value_usd'...",[True],"[['FR', 'US']]",True,True,...,NaN,NaN,NaN,NaN,NaN,1,[0],0,First,3
1,10.1038/ng.2711,2013,en,1061-4036,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['AU'], ['AU', 'US'], ['AU', 'US'], ['AU', 'U...",True,True,...,NaN,NaN,NaN,NaN,NaN,100,[0],0,First,3
2,10.1016/j.mattod.2013.06.004,2013,en,1369-7021,hybrid,"{'value': 5300, 'currency': 'USD', 'value_usd'...",[True],[['FR']],True,True,...,NaN,NaN,NaN,NaN,NaN,1,[0],0,First,3
3,10.1038/ng.2770,2013,en,1061-4036,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['US'], ['US'], ['GB', 'US'], ['US'], ['GB', ...",True,True,...,NaN,NaN,NaN,NaN,NaN,100,[0],0,First,3
4,10.1038/ngeo1797,2013,en,1752-0894,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['PK'], ['US'], ['ET', 'US'], ['IN', 'US'], [...",True,True,...,NaN,NaN,NaN,NaN,NaN,88,[0],0,First,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295306,10.1016/j.virol.2022.02.006,2022,en,0042-6822,closed,NaN,"[False, False, False, False, False, False, True]","[['EG'], ['KR'], ['EG'], ['EG'], ['KR'], ['RU'...",NaN,NaN,...,Elsevier I,Read + APC remisés,NaN,NaN,True,7,[6],6,Last,2
295307,10.1016/j.scad.2022.08.012,2022,fr,0183-2980,closed,NaN,[True],[['FR']],NaN,NaN,...,NaN,NaN,NaN,NaN,True,1,[0],0,First,3
295308,10.1016/j.scad.2022.08.015,2022,fr,0183-2980,closed,NaN,[True],[['FR']],NaN,NaN,...,NaN,NaN,NaN,NaN,True,1,[0],0,First,3
295309,10.1016/j.scad.2022.08.014,2022,fr,0183-2980,closed,NaN,[True],[['FR']],NaN,NaN,...,NaN,NaN,NaN,NaN,True,1,[0],0,First,3


### Inference method based on journals

In [27]:
threshold_mid = 0.20
threshold = 0.30
threshold_authors = 0.75
threshold_journals = 30 # Three papers per year for 10 years, which is a reasonable threshold to have a journal with a consistent CA position pattern

print(len(df_corr_fr[df_corr_fr.position == 'Middle']), len(df_corr_fr[df_corr_fr.position == 'Middle']) / len(df_corr_fr), len(df_corr_fr[(df_corr_fr.position == 'Middle') & (df_corr_fr.apc_paid.notna())]))

grouped_journal = df_corr_fr.groupby(['issn_l', 'position']).size().unstack().fillna(0)
grouped_journal['Total'] = grouped_journal.sum(axis = 1)
grouped_journal['First_pct'] = grouped_journal['First'] / grouped_journal['Total']
grouped_journal['Last_pct'] = grouped_journal['Last'] / grouped_journal['Total']
grouped_journal['Middle_pct'] = grouped_journal['Middle'] / grouped_journal['Total']
grouped_journal['entropy'] = grouped_journal.apply(lambda x: -sum((x[pct] * np.log2(x[pct]) for pct in ['First_pct', 'Last_pct', 'Middle_pct'] if x[pct] > 0)) / np.log2(3), axis = 1)

df_not_corr = df_initial[~df_initial.doi.isin(df_corr_fr.doi)].reset_index(drop = True)
df_not_corr['countries'] = df_not_corr['countries'].apply(safe_literal_eval)
df_not_corr['First_Last_FR'] = df_not_corr.countries.apply(lambda x: bool(x) and ('FR' in x[0] or 'FR' in x[-1]))
df_not_corr_fr = df_not_corr[df_not_corr.First_Last_FR == True].reset_index(drop = True) # TO DISCARD THE ONES WITH NO FRENCH IN FIRST OR LAST
print(len(df_not_corr_fr), 'Discard for inference:', len(df_not_corr[df_not_corr['First_Last_FR'] == False]))

discard_mid = grouped_journal[grouped_journal['Middle_pct'] > threshold_mid] # DISCARD MIDDLE HIGHER THAN THRESHOLD
df_not_corr_fr_low_mid = df_not_corr_fr[~df_not_corr_fr.issn_l.isin(discard_mid.index)].reset_index(drop = True)
print(len(df_not_corr_fr_low_mid), df_not_corr_fr_low_mid.apc_paid.notna().sum())

discard_ent = grouped_journal[grouped_journal['entropy'] > threshold]
df_not_corr_fr_low_mid_low_ent = df_not_corr_fr_low_mid[~df_not_corr_fr_low_mid.issn_l.isin(discard_ent.index)].reset_index(drop = True)  # THE ONES TO INFER
df_survive = df_not_corr_fr_low_mid[df_not_corr_fr_low_mid.issn_l.isin(discard_ent.index)].reset_index(drop = True)  # THE ONES FROM WHICH WE SHOULD THINK ANOTHER OPTION
print(len(df_not_corr_fr_low_mid_low_ent), len(df_survive), df_not_corr_fr_low_mid_low_ent.apc_paid.notna().sum())

df_survive_not_frlang = df_survive[df_survive.language != 'fr'].reset_index(drop = True)
df_survive_fr = df_survive[df_survive.language == 'fr'].reset_index(drop = True)
print('French language: ', len(df_survive_fr))
df_survive_not_frlang['Number_of_FR'] = df_survive_not_frlang['countries'].apply(lambda x: sum(sublist.count('FR') for sublist in x))
df_survive_not_frlang['Number_authors'] = df_survive_not_frlang['countries'].apply(lambda x: len(x))
df_survive_not_frlang['Ratio'] = df_survive_not_frlang['Number_of_FR'] / df_survive_not_frlang['Number_authors']
df_survive_not_frlang_most_fr = df_survive_not_frlang[df_survive_not_frlang['Ratio'] > threshold_authors].reset_index(drop = True)
print('Added papers to infer: ', len(df_survive_not_frlang_most_fr), df_survive_not_frlang_most_fr.apc_paid.notna().sum())

df_inferred = pd.concat([df_not_corr_fr_low_mid_low_ent, df_survive_fr, df_survive_not_frlang_most_fr.drop(columns = ['Number_of_FR', 'Number_authors', 'Ratio'])], ignore_index = True) \
                  .drop_duplicates('doi').reset_index(drop = True)
df_inferred = df_inferred.merge(grouped_journal[['First_pct', 'Last_pct', 'Middle_pct']], on = 'issn_l')
print('Inferred papers based on journals on CA: ', len(df_inferred))
df_inferred = df_inferred[df_inferred.groupby('issn_l')['issn_l'].transform('count') > threshold_journals]
print('Inferred papers based on journals on CA (after threshold): ', len(df_inferred))


9164 0.031031692012827156 2445
1177511 Discard for inference: 579439
1157884 163747
830309 327575 123706
French language:  58591
Added papers to infer:  150487 18591
Inferred papers based on journals on CA:  835571
Inferred papers based on journals on CA (after threshold):  737244


In [30]:
df_main = pd.concat((df_corr_fr.drop(columns = ['CA_positions', 'len']), df_inferred), ignore_index = True).drop_duplicates('doi').reset_index(drop = True)
print('Final number of papers configuring the main dataset: ', len(df_main))


# To include the publisher according to MESRE work
df_main['doi_prefix'] = df_main.doi.apply(lambda x: x.split('/')[0])
publisher = pd.read_csv('../data/processed/publisher_doi_full.csv')

df_publisher = df_main.merge(publisher, on = 'doi_prefix', how = 'left')
df_group = pd.read_csv('../data/processed/publisher_group.csv', header = None)[1::]
df_group.columns = df_group.iloc[0]
df_group = df_group.iloc[1:].reset_index(drop=True)[['publisher_clean', 'publisher_group', 'group_start_date']]
pub_dict = df_group.set_index('publisher_clean')['publisher_group'].to_dict()
year_dict = df_group.set_index('publisher_clean')['group_start_date'].astype(int).to_dict()
mask = (df_publisher['publisher'].isin(pub_dict) & df_publisher['publisher'].isin(year_dict) & (df_publisher['publication_year'] >= df_publisher['publisher'].map(year_dict)))
df_publisher.loc[mask, 'publisher'] = df_publisher.loc[mask, 'publisher'].map(pub_dict)
df_publisher = df_publisher.drop(columns = 'doi_prefix')
df_publisher.to_csv('../data/processed/main_dataset.csv', index = False)
df_publisher

Final number of papers configuring the main dataset:  1032555


,doi,publication_year,language,issn_l,oa_status,apc_paid,corresponding,countries,has_fr_corresponding,BSO,...,APC_Accounting_Value,Is_Couperin,keep_CA_position,position,priority,First_Last_FR,First_pct,Last_pct,Middle_pct,publisher
0,10.3389/fnins.2013.00267,2013,en,1662-453X,gold,"{'value': 2950, 'currency': 'USD', 'value_usd'...",[True],"[['FR', 'US']]",True,True,...,NaN,NaN,0.0,First,3.0,NaN,NaN,NaN,NaN,Frontiers Media SA
1,10.1038/ng.2711,2013,en,1061-4036,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['AU'], ['AU', 'US'], ['AU', 'US'], ['AU', 'U...",True,True,...,NaN,NaN,0.0,First,3.0,NaN,NaN,NaN,NaN,Springer Science and Business Media LLC
2,10.1016/j.mattod.2013.06.004,2013,en,1369-7021,hybrid,"{'value': 5300, 'currency': 'USD', 'value_usd'...",[True],[['FR']],True,True,...,NaN,NaN,0.0,First,3.0,NaN,NaN,NaN,NaN,Elsevier BV
3,10.1038/ng.2770,2013,en,1061-4036,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['US'], ['US'], ['GB', 'US'], ['US'], ['GB', ...",True,True,...,NaN,NaN,0.0,First,3.0,NaN,NaN,NaN,NaN,Springer Science and Business Media LLC
4,10.1038/ngeo1797,2013,en,1752-0894,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['PK'], ['US'], ['ET', 'US'], ['IN', 'US'], [...",True,True,...,NaN,NaN,0.0,First,3.0,NaN,NaN,NaN,NaN,Springer Science and Business Media LLC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1032550,10.1111/cod.14778,2025,en,0105-1873,bronze,NaN,[True],[[FR]],NaN,True,...,NaN,NaN,NaN,NaN,NaN,True,0.636364,0.318182,0.045455,Wiley
1032551,10.1111/hae.15148,2025,it,1351-8216,closed,NaN,"[True, False, False, False, False]","[[FR], [FR], [FR], [FR], [FR]]",NaN,True,...,NaN,NaN,NaN,NaN,NaN,True,0.585366,0.292683,0.121951,Wiley
1032552,10.1002/cpe.70008,2025,en,1532-0626,closed,NaN,[True],[[FR]],NaN,True,...,NaN,NaN,NaN,NaN,NaN,True,0.714286,0.142857,0.142857,Wiley
1032553,10.1016/j.physletb.2025.139452,2025,en,0370-2693,diamond,NaN,[True],[[FR]],NaN,True,...,NaN,NaN,NaN,NaN,NaN,True,0.758621,0.103448,0.137931,Elsevier BV
